# ⚙️ Optimización de Hiperparámetros — FIDE Chess Dataset

**Evaluación 2 — Optimización (30%)**

Este notebook cubre:
1. GridSearchCV para Random Forest y Gradient Boosting
2. Visualización del impacto de cada hiperparámetro
3. Comparación antes vs después de la optimización
4. Justificación técnica de las decisiones

In [ ]:
%load_ext kedro.ipython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42

In [ ]:
df = catalog.load('fide_preprocessed_data')

FEATURE_COLS = ['rating_std_avg', 'rating_change', 'total_months_active',
                'age_approx', 'gender_encoded', 'title_encoded']
TARGET = 'is_expert'

available = [c for c in FEATURE_COLS if c in df.columns]
ml_data = df[available + [TARGET]].dropna()
X = ml_data[available]
y = ml_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## 1. Modelo Base (sin optimizar)

Primero establecemos los resultados con parámetros por defecto.

In [ ]:
# Random Forest base
rf_base = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

print('=== Random Forest (Base) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_base):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_base, zero_division=0):.4f}')
print()
print(classification_report(y_test, y_pred_base, zero_division=0))

## 2. GridSearchCV — Random Forest

Exploramos combinaciones de `n_estimators`, `max_depth`, `min_samples_split` y `min_samples_leaf`.

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_rf.fit(X_train, y_train)

print(f'\nMejor F1 (CV): {grid_rf.best_score_:.4f}')
print(f'Mejores parámetros: {grid_rf.best_params_}')

In [ ]:
# Impacto de max_depth en el rendimiento
results_df = pd.DataFrame(grid_rf.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Impacto de max_depth
for n_est in [50, 100, 200]:
    mask = results_df['param_n_estimators'] == n_est
    subset = results_df[mask].groupby('param_max_depth')['mean_test_score'].mean()
    axes[0].plot(subset.index.astype(str), subset.values, marker='o', label=f'n_est={n_est}')

axes[0].set_title('Impacto de max_depth en F1-Score')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('F1-Score (CV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Impacto de n_estimators
for depth in [5, 10, 20]:
    mask = results_df['param_max_depth'] == depth
    subset = results_df[mask].groupby('param_n_estimators')['mean_test_score'].mean()
    axes[1].plot(subset.index, subset.values, marker='s', label=f'depth={depth}')

axes[1].set_title('Impacto de n_estimators en F1-Score')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('F1-Score (CV)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Análisis de Hiperparámetros \u2014 Random Forest', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. GridSearchCV — Gradient Boosting

In [ ]:
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_samples_split': [2, 5],
}

grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid_gb,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_gb.fit(X_train, y_train)

print(f'\nMejor F1 (CV): {grid_gb.best_score_:.4f}')
print(f'Mejores parámetros: {grid_gb.best_params_}')

In [ ]:
# Impacto del learning_rate
gb_results = pd.DataFrame(grid_gb.cv_results_)

fig, ax = plt.subplots(figsize=(10, 6))
for n_est in [50, 100, 200]:
    mask = gb_results['param_n_estimators'] == n_est
    subset = gb_results[mask].groupby('param_learning_rate')['mean_test_score'].mean()
    ax.plot(subset.index, subset.values, marker='o', label=f'n_est={n_est}')

ax.set_title('Impacto del Learning Rate en Gradient Boosting')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('F1-Score (CV)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Comparación: Antes vs Después de la Optimización

In [ ]:
# Evaluar modelos optimizados en test
y_pred_rf_opt = grid_rf.best_estimator_.predict(X_test)
y_pred_gb_opt = grid_gb.best_estimator_.predict(X_test)

comparison = pd.DataFrame({
    'RF Base': {
        'Accuracy': accuracy_score(y_test, y_pred_base),
        'F1-Score': f1_score(y_test, y_pred_base, zero_division=0),
    },
    'RF Optimizado': {
        'Accuracy': accuracy_score(y_test, y_pred_rf_opt),
        'F1-Score': f1_score(y_test, y_pred_rf_opt, zero_division=0),
    },
    'GB Optimizado': {
        'Accuracy': accuracy_score(y_test, y_pred_gb_opt),
        'F1-Score': f1_score(y_test, y_pred_gb_opt, zero_division=0),
    },
}).T

display(comparison.style.format('{:.4f}').background_gradient(cmap='YlGn'))

# Gráfico
comparison.plot.bar(figsize=(10, 6), rot=0)
plt.title('Comparación: Antes vs Después de Optimización')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Conclusiones de la Optimización

- GridSearchCV permitió explorar sistemáticamente el espacio de hiperparámetros.
- Se visualizó el impacto individual de `max_depth`, `n_estimators` y `learning_rate`.
- La optimización mejora (o al menos mantiene) el rendimiento respecto al modelo base.
- La reproduciblidad está garantizada con `random_state=42`.